#### 1. Configurations

Builds the offline `sdw` package (webui + its 5 pinned sub-repos + a chosen set of extensions, see `build_package.py`) and optionally uploads it to HuggingFace. Default models/VAEs are not bundled in the package - `auto_launcher.ipynb`'s own `Params.links` is the source of truth for what a fresh `links.json` gets seeded with.

- **HuggingFace write token**: needs **write** scope (different from the read-only token used in `auto_launcher.ipynb`) - get one at https://huggingface.co/settings/tokens
- **Upload immediately**: checked = build the archive and upload it to the HF repo below in one go. Unchecked = build the archive only, so you can inspect it before uploading (upload later by re-running with the box checked, or by running `build_package.upload(...)` manually).

In [ ]:
# Run to set up the build configuration
import importlib, sys
from pathlib import Path
from os import getenv
from ipywidgets import Text, Password, Checkbox, Dropdown, Textarea, Layout
from IPython.display import display

import build_package
importlib.reload(build_package)

style = {"description_width": "25%"}
layout = {"width": "70%"}

hf_write_token_control = Password(
    value=getenv("HF_WRITE_TOKEN", ""),
    description="HF write token",
    placeholder="hf_... (write-scoped, not the read-only one)",
    style=style,
    layout=layout,
)

hf_repo_control = Text(
    value="",
    description="HF repo id",
    placeholder="your-username/sdwebui-package",
    style=style,
    layout=layout,
)

repo_type_control = Dropdown(
    options=["dataset", "model"],
    value="dataset",
    description="HF repo type",
    style=style,
    layout=layout,
)

output_control = Text(
    value="sdw_package.tar.xz",
    description="Output archive path",
    style=style,
    layout=layout,
)

scratch_dir_control = Text(
    value="./_package_build",
    description="Scratch dir",
    style=style,
    layout=layout,
)

extensions_control = Textarea(
    value="\n".join(f"{name}={url}" for name, url in build_package.EXTENSIONS.items()),
    description="Extensions",
    placeholder="name=https://github.com/...  (one per line)",
    style=style,
    layout=Layout(width="70%", height="120px"),
)

upload_immediately_control = Checkbox(
    value=True,
    description="Upload to HuggingFace immediately after building",
    layout=layout,
    style=style,
)

display(
    hf_write_token_control,
    hf_repo_control,
    repo_type_control,
    output_control,
    scratch_dir_control,
    extensions_control,
    upload_immediately_control,
)

### 2. Build 🚀

In [ ]:
# Run this when you're ready
class StopExecution(Exception):
    def _render_traceback_(self):
        pass
try:
    hf_write_token_control
except NameError:
    display("Please run the configurations block")
    raise StopExecution

# apply the editable extensions list from the UI
extensions = {}
for line in extensions_control.value.strip().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    name, url = line.split("=", 1)
    extensions[name.strip()] = url.strip()
if extensions:
    build_package.EXTENSIONS = extensions

scratch_dir = Path(scratch_dir_control.value).resolve()
scratch_dir.mkdir(parents=True, exist_ok=True)
output = Path(output_control.value).resolve()

display(f"Building package into {output} ...")
sdw_dir = build_package.build(scratch_dir)
build_package.pack_package(sdw_dir, output)

if upload_immediately_control.value:
    hf_token = hf_write_token_control.value.strip()
    hf_repo = hf_repo_control.value.strip()
    if not hf_token or not hf_repo:
        display("Set the HF repo id and write token above before uploading - skipping upload.")
    else:
        build_package.upload(output, hf_repo, hf_token, repo_type_control.value)
        display("Done. Paste the download URL above into auto_launcher.ipynb's OFFLINE_PACKAGE_URL constant.")
else:
    display(f"Built {output} - upload skipped (box unchecked). Check the box and re-run this cell to upload it, or run build_package.upload(output, '<repo>', '<token>') manually.")